# Challenge Quotidien : Prétraitement et Fine-tuning de modèles Transformers

Ce notebook documente le processus de préparation de modèles BERT et XLM-RoBERTa pour la classification de texte. Nous allons suivre un workflow structuré pour comprendre comment ces modèles traitent le langage humain.

In [1]:
# Installation des bibliothèques nécessaires
!pip install -q transformers sentencepiece

## 1. Compréhension de BERT et XLM-RoBERTa

- **BERT** : Conçu pour comprendre le contexte bidirectionnel dans les textes en anglais.
- **XLM-RoBERTa** : Une version multilingue robuste capable de traiter plus de 100 langues différentes.

In [2]:
from transformers import BertTokenizer, XLMRobertaTokenizer

# Initialisation des tokenizers avec des modèles pré-entraînés de base
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

print("Tokenizers chargés avec succès.")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Tokenizers chargés avec succès.


## 2 & 3. Tokenisation et Préparation des données

Nous allons transformer une phrase en 'input_ids' (nombres représentatifs) et 'attention_mask' (pour ignorer le remplissage/padding).

In [6]:
sentence = "L'apprentissage automatique est fascinant !"

# Utilisation de la méthode d'appel direct (recommandée par Hugging Face)
# Cela remplace encode_plus et gère automatiquement le formatage
encoded_input = bert_tokenizer(
    sentence,
    add_special_tokens=True,     # Ajoute [CLS] au début et [SEP] à la fin
    max_length=15,               # Définit une longueur maximale de 15 tokens
    padding='max_length',        # Remplit avec des tokens vides si la phrase est courte
    truncation=True,             # Coupe la phrase si elle dépasse 15 tokens
    return_attention_mask=True,  # Crée le masque pour que le modèle ignore le padding
    return_tensors='pt'          # Retourne les données sous forme de tenseurs PyTorch
)

print(f"Phrase originale : {sentence}")
print(f"ID des tokens (input_ids) : \n{encoded_input['input_ids']}")
print(f"Masque d'attention (attention_mask) : \n{encoded_input['attention_mask']}")

# Décodage pour vérifier ce que le modèle 'voit'
print(f"Texte décodé : {bert_tokenizer.decode(encoded_input['input_ids'][0])}")

Phrase originale : L'apprentissage automatique est fascinant !
ID des tokens (input_ids) : 
tensor([[  101,  1048,  1005, 10439, 22787, 21205,  3351,  8285, 18900,  7413,
          9765,  6904, 11020,  3981,   102]])
Masque d'attention (attention_mask) : 
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
Texte décodé : [CLS] l ' apprentissage automatique est fascina [SEP]


## 4. Chargement et Exploration du Dataset

Nous allons extraire les fichiers ZIP et charger les fichiers CSV fournis dans l'environnement.

In [4]:
import pandas as pd
import zipfile
import os

# Extraction des fichiers si nécessaire
for zip_file in ['train.csv.zip', 'test.csv.zip']:
    if os.path.exists(f'/content/{zip_file}'):
        with zipfile.ZipFile(f'/content/{zip_file}', 'r') as zip_ref:
            zip_ref.extractall('/content/')

# Chargement des DataFrames
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')

print(f"Taille du dataset d'entraînement : {train_df.shape}")
display(train_df.head())

Taille du dataset d'entraînement : (12120, 6)


,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1


## 5. Création des Folds pour la Validation Croisée

La validation croisée (K-Fold) permet de s'assurer que le modèle est stable sur différentes parties des données.

In [5]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

# On suppose que la colonne cible s'appelle 'label' (à ajuster selon le dataset)
target_column = 'label' if 'label' in train_df.columns else train_df.columns[-1]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

folds = []

# Séparation en 5 folds
for fold_num, (train_index, val_index) in enumerate(skf.split(train_df, train_df[target_column])):
    train_fold = train_df.iloc[train_index]
    val_fold = train_df.iloc[val_index]
    folds.append((train_fold, val_fold))
    print(f"Fold {fold_num + 1} créé. Train: {len(train_index)}, Val: {len(val_index)}")

Fold 1 créé. Train: 9696, Val: 2424
Fold 2 créé. Train: 9696, Val: 2424
Fold 3 créé. Train: 9696, Val: 2424
Fold 4 créé. Train: 9696, Val: 2424
Fold 5 créé. Train: 9696, Val: 2424
